### 2D view for models in SASView
### theory only with an empty 2D data set
### comparision of rectangular prism with prism model with n=4

run using global (Python 3.12.10) on PC fixe 

import all the packages

In [ ]:
import numpy as np

import sasmodels

import sasmodels.core
import sasmodels.data
import sasmodels.bumps_model

import sasdata

from sasdata.dataloader.loader import Loader
from sasmodels.data import load_data
#from sas import sascalc.dataloader

from sasmodels.data import plot_theory

import bumps
import bumps.fitters
import bumps.names
import bumps.fitproblem

import os



In [ ]:

"""
import matplotlib.pyplot and set custom default settings for plots
"""
%matplotlib inline
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')


import matplotlib.pyplot as plt
#from IPython.display import set_matplotlib_formats
#set_matplotlib_formats('svg')

plt.rc('font', size=14) # default fontsize
plt.rc('axes', titlesize=16)
plt.rc('axes', labelsize=16)
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
plt.rc('legend', fontsize=14)
plt.rc('legend', title_fontsize=14)
plt.rc('figure', titlesize=18)
plt.rc("figure", figsize=(10,10))
plt.rc("lines", linewidth=3)

#import tol_colors as tc # colorblind safe color palettes

# use colorblind safe colors:
#plt.rc('axes', prop_cycle=plt.cycler('color', list(tc.tol_cset('muted'))))
#try:
#    plt.cm.register_cmap('tc_iridescent', tc.tol_cmap('iridescent'))
#except:
#    pass

#plt.rc('image', cmap='tc_iridescent')

#plt.rc('image', cmap='rainbow')
plt.rc('image', cmap='jet')



## import radius_from_edge from the prism model

In [ ]:
from sasmodels.models.prism import radius_from_edge # circumradius

from sasmodels.models.prism import edge_from_gyration_radius #average radius

### Calling a model with an empty data set

In [ ]:
q_max = 0.4 # limit along edge of detector
num_q = 100 # number of q values along x and y directions, use even number
q = np.linspace(q_max*0.5/(int(num_q/2)-0.5), q_max, int(num_q/2))
q = np.hstack((-1*np.flip(q), q))
data2d = sasmodels.data.empty_data2D(qx=q, qy=q, resolution=0)

In [ ]:
sld_gold=125
sld_silver=77.9
sld_water=9.47
sld_toluene=8.4

In [ ]:
phi_angle=20
theta_angle=30
psi_angle=40

In [ ]:
# average radius and length for the prism model
ave_rad=50
length_value=400

# corresponding dimensions for the rectangular_prism model
a_value=edge_from_gyration_radius(4,ave_rad)
b_value=a_value
c_value=length_value


### rectangular prism

In [ ]:
modelname1="rectangular_prism"

In [ ]:
kernel1 = sasmodels.core.load_model(modelname1)
call_kernel = sasmodels.direct_model.DirectModel(data2d, kernel1)
kernel1.info.parameters.defaults

In [ ]:
Iq1 = call_kernel(
    phi=phi_angle, theta=theta_angle, psi=psi_angle,
    sld=sld_gold,sld_solvent=sld_water,
    length_a=a_value,
    b2a_ratio=b_value/a_value,
    c2a_ratio=c_value/a_value,
#    psi_pd_type='uniform', #polydispersity function
#    psi_pd=180, #polydispersity
#    psi_pd_n=10, #number of steps in polydispersity
)

fig, ax = plt.subplots()
im1 = ax.pcolormesh(data2d.qx_data.reshape(2*int(num_q/2), -1), data2d.qy_data.reshape(2*int(num_q/2), -1), np.log10(Iq1.reshape(2*int(num_q/2), -1)))
fig.colorbar(im1, label='log$_{10}$[I(q)]')

ratio = 1.0
x_min, x_max = (np.min(data2d.qx_data), np.max(data2d.qx_data))
y_min, y_max = (np.min(data2d.qy_data), np.max(data2d.qy_data))
ax.set_aspect(abs((x_max-x_min)/(y_min-y_max))*ratio)

plt.xlabel("q$_x$ ($A^{-1}$)")
plt.ylabel("q$_y$ ($A^{-1}$)")

#plt.show()

### prism model

In [ ]:
modelname2="prism"

In [ ]:
kernel2 = sasmodels.core.load_model(modelname2)
call_kernel = sasmodels.direct_model.DirectModel(data2d, kernel2)
kernel2.info.parameters.defaults

In [ ]:
Iq2 = call_kernel(phi=phi_angle, theta=theta_angle, psi=psi_angle-45,
                  sld=sld_gold,sld_solvent=sld_water,
                  n_sides=4,
                  radius_average=ave_rad,
                  length=length_value,
#                  psi_pd_type='uniform', #polydispersity function
 #                 psi_pd=180, #polydispersity
 #                 psi_pd_n=10 #number of steps in polydispersity
)

fig, ax = plt.subplots()
im2 = ax.pcolormesh(data2d.qx_data.reshape(2*int(num_q/2), -1), data2d.qy_data.reshape(2*int(num_q/2), -1), np.log10(Iq2.reshape(2*int(num_q/2), -1)))
fig.colorbar(im2, label='log$_{10}$[I(q)]')

ratio = 1.0
x_min, x_max = (np.min(data2d.qx_data), np.max(data2d.qx_data))
y_min, y_max = (np.min(data2d.qy_data), np.max(data2d.qy_data))
ax.set_aspect(abs((x_max-x_min)/(y_min-y_max))*ratio)

plt.xlabel("q$_x$ ($A^{-1}$)")
plt.ylabel("q$_y$ ($A^{-1}$)")

#plt.show()

In [ ]:
fig12, (ax1, ax2) = plt.subplots(1, 2)
fig12.suptitle('rectangular_prism                 prism with n=4')

im1 = ax1.pcolormesh(data2d.qx_data.reshape(2*int(num_q/2), -1), data2d.qy_data.reshape(2*int(num_q/2), -1), np.log10(Iq1.reshape(2*int(num_q/2), -1)))
#fig12.colorbar(im1, label='log$_{10}$[I(q)]')

ratio = 1.0
x_min, x_max = (np.min(data2d.qx_data), np.max(data2d.qx_data))
y_min, y_max = (np.min(data2d.qy_data), np.max(data2d.qy_data))
ax1.set_aspect(abs((x_max-x_min)/(y_min-y_max))*ratio)

im2 = ax2.pcolormesh(data2d.qx_data.reshape(2*int(num_q/2), -1), data2d.qy_data.reshape(2*int(num_q/2), -1), np.log10(Iq2.reshape(2*int(num_q/2), -1)))
#fig12.colorbar(im2, label='log$_{10}$[I(q)]')

x_min, x_max = (np.min(data2d.qx_data), np.max(data2d.qx_data))
y_min, y_max = (np.min(data2d.qy_data), np.max(data2d.qy_data))
ax2.set_aspect(abs((x_max-x_min)/(y_min-y_max))*ratio)


plt.xlabel("q$_x$ ($A^{-1}$)")
plt.ylabel("q$_y$ ($A^{-1}$)")
